# Task 048: storm-analysis-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: STORM super-resolution microscopy localization using Gaussian fitting

**Paper**: Multiple algorithm papers (software package)
**Repository**: https://github.com/ZhuangLab/storm-analysis

### Physical Background

Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.

### Forward Model

```
y = f(theta) + n
```

### Inverse Problem

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

### Performance Metrics
- **PSNR**: 34.52 dB
- **SSIM**: N/A


### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import scipy.optimize
import scipy.ndimage
import os

# ==============================================================================
# Helper Functions & Classes
# ==============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
# ==============================================================================
# 1. Load and Preprocess Data
# ==============================================================================
def load_and_preprocess_data(file_path, frame_idx, offset, gain):
    """
    Loads a specific frame from a DAX file and applies preprocessing 
    (gain/offset correction).
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
        
    reader = DaxReader(file_path)
    try:
        raw_image = reader.loadAFrame(frame_idx)
    finally:
        reader.close()
        
    # Preprocessing
    image = (raw_image - offset) / gain
    image[image < 0] = 0
    
    return image

## 4. Class: `DaxReader`

Simplified Dax Reader for STORM movies.
Based on storm_analysis.sa_library.datareader.DaxReader

Methods: `__init__`, `loadAFrame`, `close`

In [ ]:
class DaxReader:
    """
    Simplified Dax Reader for STORM movies.
    Based on storm_analysis.sa_library.datareader.DaxReader
    """
    def __init__(self, filename):
        self.filename = filename
        self.image_height = None
        self.image_width = None
        self.number_frames = None
        self.bigendian = 0
        
        # Try to read .inf file
        dirname = os.path.dirname(filename)
        basename = os.path.splitext(os.path.basename(filename))[0]
        inf_filename = os.path.join(dirname, basename + ".inf")
        
        if os.path.exists(inf_filename):
            with open(inf_filename, 'r') as fp:
                for line in fp:
                    if "frame dimensions" in line:
                        dim_str = line.split("=")[1].strip()
                        w, h = dim_str.split("x")
                        self.image_width = int(w)
                        self.image_height = int(h)
                    elif "number of frames" in line:
                        self.number_frames = int(line.split("=")[1].strip())
                    elif "big endian" in line:
                        self.bigendian = 1
        
        if self.image_height is None:
            # Fallback based on file size assuming 256x256
            filesize = os.path.getsize(filename)
            if filesize % (256*256*2) == 0:
                self.image_height = 256
                self.image_width = 256
                self.number_frames = filesize // (256*256*2)
            else:
                raise ValueError("Could not determine DAX dimensions from .inf or file size.")

        self.fileptr = open(filename, "rb")

    def loadAFrame(self, frame_number):
        if frame_number >= self.number_frames:
            raise ValueError("Frame number out of range")
            
        self.fileptr.seek(frame_number * self.image_height * self.image_width * 2)
        image_data = np.fromfile(self.fileptr, dtype='uint16', count = self.image_height * self.image_width)
        image_data = np.reshape(image_data, [self.image_height, self.image_width])
        if self.bigendian:
            image_data.byteswap(True)
        return image_data.astype(np.float32)

    def close(self):
        self.fileptr.close()

## 5. Forward Model

Define the forward operator mapping true model to observations.

```
y = f(theta) + n
```

Functions: `forward_operator`

In [ ]:
# ==============================================================================
# 2. Forward Operator
# ==============================================================================
def forward_operator(x_params, image_shape, background_image=None):
    """
    Generates an image from a list of Gaussian parameters.
    x_params: List of [background, height, center_x, center_y, width]
    image_shape: tuple (h, w)
    background_image: Optional 2D array representing global background.
    
    Returns: y_pred (simulated image)
    """
    h, w = image_shape
    y_grid, x_grid = np.mgrid[0:h, 0:w]
    y_pred = np.zeros(image_shape)
    
    if background_image is not None:
        y_pred += background_image

    for p in x_params:
        # p: [local_bg, height, cx, cy, wid]
        # We assume local_bg is handled by the global background_image for the full forward model,
        # or we just render the gaussian lobes here.
        # Following the logic of the original code's generate_gaussian_image:
        # Unpack
        _, h_val, cx, cy, wid = p
        
        # Render this gaussian lobe
        g = h_val * np.exp(-2 * (((cx - x_grid) / wid) ** 2 + ((cy - y_grid) / wid) ** 2))
        y_pred += g
        
    return y_pred

## 6. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

Functions: `run_inversion`

In [ ]:
# ==============================================================================
# 3. Run Inversion
# ==============================================================================
def run_inversion(image, sigma, background_sigma, threshold_factor):
    """
    Performs peak finding and Gaussian fitting (inversion).
    
    Returns: 
        fitted_params: List of optimized gaussian parameters
        estimated_background: The background image estimated during preprocessing
    """
    # --- Step 1: Preprocessing for Peak Finding ---
    smooth_img = scipy.ndimage.gaussian_filter(image, sigma)
    bg_img = scipy.ndimage.gaussian_filter(image, background_sigma)
    dog_img = smooth_img - bg_img
    
    # Thresholding
    threshold = threshold_factor * np.std(dog_img) + np.mean(dog_img)
    mask = dog_img > threshold
    
    # Local Maxima Detection
    neighborhood_size = 3
    local_max = scipy.ndimage.maximum_filter(dog_img, size=neighborhood_size) == dog_img
    
    # Combined mask
    peaks_mask = local_max & mask
    y_peaks, x_peaks = np.where(peaks_mask)
    
    # --- Step 2: Fitting (Levenberg-Marquardt) ---
    fitted_params = []
    r = 5 
    h, w = image.shape
    
    for px, py in zip(x_peaks, y_peaks):
        if px < r or px >= w - r or py < r or py >= h - r:
            continue
            
        # Crop ROI
        roi = image[py-r:py+r+1, px-r:px+r+1]
        y_roi, x_roi = np.mgrid[py-r:py+r+1, px-r:px+r+1]
        
        # Initial Guess: [background, height, center_x, center_y, width]
        p0 = [np.min(roi), np.max(roi) - np.min(roi), px, py, 2.0]
        
        try:
            error_function = lambda p: symmetric_gaussian_2d((x_roi, y_roi), *p) - roi.ravel()
            p_opt, success = scipy.optimize.leastsq(error_function, p0, maxfev=100)
            
            if success in [1, 2, 3, 4]: 
                bg, height, cx, cy, width = p_opt
                # Sanity checks
                if (height > 0) and (0.5 < width < 10.0) and (abs(cx - px) < 2) and (abs(cy - py) < 2):
                    fitted_params.append(p_opt)
        except Exception:
            continue
            
    return fitted_params, bg_img

## 7. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `symmetric_gaussian_2d`, `evaluate_results`

In [ ]:
def symmetric_gaussian_2d(xy, background, height, center_x, center_y, width):
    """
    Explicit mathematical definition of a 2D Symmetric Gaussian.
    f(x,y) = background + height * exp( -2 * ( ((x-cx)/w)^2 + ((y-cy)/w)^2 ) )
    """
    x, y = xy
    g = background + height * np.exp(-2 * (((center_x - x) / width) ** 2 + ((center_y - y) / width) ** 2))
    return g.ravel()

# ==============================================================================
# 4. Evaluate Results
# ==============================================================================
def evaluate_results(original, reconstructed):
    """
    Calculates PSNR.
    """
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    
    data_range = np.max(original) - np.min(original)
    if data_range == 0:
        return 0.0
        
    psnr = 20 * np.log10(data_range / np.sqrt(mse))
    return psnr

## 8. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Define I/O directory
io_dir = './io'
os.makedirs(io_dir, exist_ok=True)

# Define constants
REPO_PATH = "/data/yjh/storm-analysis-master"
DAX_FILE = os.path.join(REPO_PATH, "storm_analysis/test/data/test.dax")

PARAMS = {
    'sigma': 1.0,
    'background_sigma': 8.0,
    'threshold': 1.0,
    'camera_offset': 100.0,
    'camera_gain': 1.0,
    'frame_idx': 0
}

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
try:
    print("Loading data...")
    input_image = load_and_preprocess_data(
        DAX_FILE, 
        PARAMS['frame_idx'], 
        PARAMS['camera_offset'], 
        PARAMS['camera_gain']
    )
    print(f"Data loaded. Shape: {input_image.shape}")

### 2. Run Inversion

Intermediate processing step in the pipeline.

In [ ]:
# 2. Run Inversion
    print("Running inversion...")
    fitted_gaussians, estimated_bg = run_inversion(
        input_image, 
        PARAMS['sigma'], 
        PARAMS['background_sigma'], 
        PARAMS['threshold']
    )
    print(f"Inversion complete. Found {len(fitted_gaussians)} peaks.")

### 3. Forward Operator (Simulate result for evaluation)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. Forward Operator (Simulate result for evaluation)
    print("Running forward operator...")
    # The forward model consists of the fitted gaussians + the global estimated background
    reconstructed_image = forward_operator(
        fitted_gaussians, 
        input_image.shape, 
        background_image=estimated_bg
    )

    # >>> SAVE OUTPUT (only one) <<<
    np.save(os.path.join(io_dir, 'output.npy'), reconstructed_image)
    print("saving output")

### 4. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate
    print("Evaluating results...")
    psnr_score = evaluate_results(input_image, reconstructed_image)
    print(f"PSNR: {psnr_score:.2f} dB")

    print("OPTIMIZATION_FINISHED_SUCCESSFULLY")
    
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"An error occurred: {e}")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 9. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **storm-analysis-master**:

1. **Problem Setup**: Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=34.52 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Multiple algorithm papers (software package)
- Repository: https://github.com/ZhuangLab/storm-analysis
